# Chapter 7: The 27 lines on a cubic surface

Source orientation: printed pages 109-120; physical PDF pages 107-118; sections 7.1-7.7 and exercises. The source span is used here for terminology, route, and formulas only. The explanations, diagrams, code, and checks are original to this notebook.

## Chapter Goal

The classical theorem says that a nonsingular cubic surface in projective 3-space contains exactly 27 lines. The computational goal is to make the statement inspectable: a projective line is represented by two spanning vectors in homogeneous coordinates, lying on a cubic is tested by substituting the line into the cubic and checking four coefficients, and incidence between two lines is tested by Plucker coordinates.

The notebook keeps two viewpoints in play. First, it follows the proof architecture for an arbitrary nonsingular cubic surface: tangent plane sections, polar forms, a pencil of planes through a known line, and a resultant whose degree records the special cases. Second, it uses the Fermat cubic

\[
X^3 + Y^3 + Z^3 + T^3 = 0 \subset \mathbf{P}^3
\]

as an exact model where all 27 lines can be written down and checked. The Fermat model is not a proof for every cubic surface by itself. It is a concrete laboratory for the same algebraic tests used in the general proof.

## Computational Translation Guide

| Geometric phrase | Computational object | Check that should fail if translated incorrectly |
| --- | --- | --- |
| A line in \(\mathbf P^3\) | Two independent vectors \(u,v\in k^4\), with points \(su+tv\) | Plucker relation \(p_{01}p_{23}-p_{02}p_{13}+p_{03}p_{12}=0\) |
| A line lies on a cubic surface | The binary cubic \(F(su+tv)\) is identically zero | Four coefficients of \(s^3,s^2t,st^2,t^3\) vanish |
| Tangency and line containment | Polar coefficients in \(F(\lambda P+\mu Q)\) | Polar expansion residual is exactly zero |
| Singular conic in a plane pencil | A discriminant or resultant condition in the plane parameter | Sylvester degree bookkeeping gives the expected degree 27 step |
| Two projective lines meet | The 4-vector wedge of their two Plucker 2-vectors vanishes | Incidence graph has degree 10 at every Fermat line |
| Nonsingular Fermat cubic | The gradient has no nonzero projective common zero | All partials vanish only at the forbidden zero vector |

## Refreshed Visual Storyboard

1. `cubic-surface-plane-sections.png`: a plane through a known line cuts the cubic as that line plus a residual conic; the singular residual cases split into two more lines.
2. `polar-line-condition-flow.png`: the polar identity turns a line-containment question into four scalar equations.
3. `resultant-degree-sylvester.png`: the elimination step is a Sylvester determinant with degree labels, showing the degree-27 leading term.
4. `fermat-cubic-lines.png`: the Fermat cubic supplies 3 pairings of coordinates and 9 root choices per pairing.
5. `twenty-seven-line-incidence-graph.png`: Plucker incidence converts the 27 lines into a graph with uniform degree 10.
6. `twenty-seven-line-incidence-graph-lab.html`: a Plotly lab projects complex Fermat lines to a real 3D chart for hover inspection.

The final sanity cell writes JSON checks under `artifacts/chapter-07/checks/` using book-local paths such as `artifacts/chapter-07/figures/...`.


In [ ]:
from pathlib import Path
import itertools
import json
import sys

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import sympy as sp
from IPython.display import Markdown, display


def find_book_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (
            (candidate / "AGENTS.md").exists()
            and (candidate / "scripts" / "validate_uag_course.py").exists()
            and (candidate / "utils").exists()
        ):
            return candidate
    fallback = start / "Undergraduate-Algebraic-Geometry"
    if (fallback / "AGENTS.md").exists() and (fallback / "utils").exists():
        return fallback
    raise RuntimeError("Could not locate Undergraduate-Algebraic-Geometry course root")


BOOK_ROOT = find_book_root(Path.cwd())
ARTIFACT_ROOT = BOOK_ROOT / "artifacts" / "chapter-07"
FIGURE_ROOT = ARTIFACT_ROOT / "figures"
HTML_ROOT = ARTIFACT_ROOT / "html"
CHECK_ROOT = ARTIFACT_ROOT / "checks"
TABLE_ROOT = ARTIFACT_ROOT / "tables"
for folder in [FIGURE_ROOT, HTML_ROOT, CHECK_ROOT, TABLE_ROOT]:
    folder.mkdir(parents=True, exist_ok=True)

if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))
if str(BOOK_ROOT / "scripts") not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT / "scripts"))

from utils.artifacts import assert_artifacts, display_artifact
from utils.validation import validate_chapter_outputs


def book_local(path: Path) -> str:
    return Path(path).resolve().relative_to(BOOK_ROOT).as_posix()


storyboard = {
    "chapter": 7,
    "title": "The 27 lines on a cubic surface",
    "source_span": {"printed": "109-120", "pdf": "107-118"},
    "goal": "Make line containment, polar equations, resultants, and Plucker incidence executable for cubic surfaces.",
    "visual_sequence": [
        "cubic-surface-plane-sections.png",
        "polar-line-condition-flow.png",
        "resultant-degree-sylvester.png",
        "fermat-cubic-lines.png",
        "twenty-seven-line-incidence-graph.png",
        "twenty-seven-line-incidence-graph-lab.html",
    ],
    "libraries": {
        "sympy": "exact homogeneous-coordinate identities, polar forms, resultants, and Plucker checks",
        "matplotlib": "durable 2D proof diagrams and degree/incidence figures",
        "networkx": "incidence graph and proof-flow graph",
        "plotly": "interactive 3D chart probe for Fermat line representatives",
        "pandas": "book-local CSV ledgers for audits and inspection",
    },
}
(CHECK_ROOT / "storyboard.json").write_text(json.dumps(storyboard, indent=2) + "\n", encoding="utf-8")
print(f"BOOK_ROOT = {BOOK_ROOT}")
print(f"chapter artifacts = {book_local(ARTIFACT_ROOT)}")


## 1. Plane Sections Through a Line

A useful entry point is not the full cubic surface at once, but a plane section. If a plane contains a line \(\ell\subset S\), then the cubic equation restricted to that plane is divisible by the equation of \(\ell\). The residual factor is a conic. For most planes in the pencil through \(\ell\), this residual conic is smooth. For special planes it becomes singular and splits into two more lines on the surface.

This is the local mechanism behind the five pairs of lines meeting a fixed line. Nonsingularity matters here: a doubled residual line would force a singular point on the cubic surface, so the split cases are simple. The figure records the two states of the plane pencil rather than trying to draw the whole projective surface in one picture.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), constrained_layout=True)

ax = axes[0]
theta = np.linspace(0, 2 * np.pi, 300)
ax.plot([-1.25, 1.25], [-1.0, -1.0], color="#263238", lw=3, label="fixed line l")
ax.plot(0.78 * np.cos(theta), 0.42 * np.sin(theta) + 0.18, color="#0072B2", lw=2.5, label="residual conic")
ax.text(-1.18, -1.25, "l", fontsize=13, weight="bold")
ax.text(-0.33, 0.72, "smooth residual conic", color="#0072B2", fontsize=11)
ax.set_title("generic plane in the pencil")
ax.set_aspect("equal")
ax.set_xlim(-1.45, 1.45)
ax.set_ylim(-1.45, 1.15)
ax.axis("off")

ax = axes[1]
ax.plot([-1.25, 1.25], [-1.0, -1.0], color="#263238", lw=3)
ax.plot([-1.1, 1.1], [0.95, -0.75], color="#D55E00", lw=2.5)
ax.plot([-1.1, 1.1], [-0.75, 0.95], color="#009E73", lw=2.5)
ax.plot([0], [0.1], "o", color="#111111", ms=5)
ax.text(-1.2, -1.25, "l", fontsize=13, weight="bold")
ax.text(0.62, -0.66, "l_i", color="#D55E00", fontsize=12)
ax.text(0.62, 0.66, "l'_i", color="#009E73", fontsize=12)
ax.text(-1.32, 0.82, "split residual conic", fontsize=11)
ax.set_title("one of five singular planes")
ax.set_aspect("equal")
ax.set_xlim(-1.45, 1.45)
ax.set_ylim(-1.45, 1.15)
ax.axis("off")

fig.suptitle("A plane through a line cuts the cubic as l plus a residual conic", fontsize=14, weight="bold")
plane_section_path = FIGURE_ROOT / "cubic-surface-plane-sections.png"
fig.savefig(plane_section_path, dpi=170, bbox_inches="tight")
plt.close(fig)
display_artifact(plane_section_path, width=760)


## 2. Line Substitution and the Polar Form

Let \(F\) be a homogeneous cubic and let a projective line be spanned by \(P,Q\). The line lies on the cubic surface exactly when \(F(\lambda P+\mu Q)\) is the zero binary cubic. Equivalently, its four coefficients vanish. The polar form packages those four coefficients as

\[
F(\lambda P+\mu Q)=\lambda^3F(P)+\lambda^2\mu F_1(P;Q)+\lambda\mu^2F_1(Q;P)+\mu^3F(Q),
\]

where \(F_1(P;Q)=\sum_i (\partial F/\partial X_i)(P)Q_i\). Thus a proposed secant line becomes a four-equation algebraic test. The code verifies the identity for a cuspidal tangent-section normal form and extracts the equations \(A,B,C\) used after choosing \(P\in S\) and \(Q\) in a complementary plane.


In [ ]:
X, Y, Z, T = sp.symbols("X Y Z T")
lam, mu, alpha = sp.symbols("lambda mu alpha")
qY, qZ, qT = sp.symbols("qY qZ qT")
variables = [X, Y, Z, T]

g = Z**2 + X * T + Y * Z + T**2
F_cusp = X**2 * Z - Y**3 + g * T
P_alpha = [1, alpha, alpha**3, 0]
Q_plane = [0, qY, qZ, qT]


def eval_poly(poly, point):
    return sp.expand(poly.subs(dict(zip(variables, point))))


def polar(poly, at_point, toward_point):
    return sp.expand(
        sum(sp.diff(poly, var).subs(dict(zip(variables, at_point))) * toward_point[i] for i, var in enumerate(variables))
    )


A = sp.factor(polar(F_cusp, P_alpha, Q_plane))
B = sp.factor(polar(F_cusp, Q_plane, P_alpha))
C = sp.factor(eval_poly(F_cusp, Q_plane))
line_point = [lam * P_alpha[i] + mu * Q_plane[i] for i in range(4)]
polar_identity_residual = sp.simplify(
    sp.expand(eval_poly(F_cusp, line_point))
    - (lam**3 * eval_poly(F_cusp, P_alpha) + lam**2 * mu * A + lam * mu**2 * B + mu**3 * C)
)
assert polar_identity_residual == 0
display(Markdown("\n".join([f"- `{name}`: `{sp.sstr(expr)}`" for name, expr in {
    "A = F1(P_alpha;Q)": A,
    "B = F1(Q;P_alpha)": B,
    "C = F(Q)": C,
}.items()])))
polar_identity_residual


In [ ]:
G = nx.DiGraph()
G.add_edges_from([
    ("choose P on S", "F(P)=0"),
    ("choose Q in plane", "F(Q)=0"),
    ("polar at P", "F1(P;Q)=0"),
    ("polar at Q", "F1(Q;P)=0"),
    ("F(P)=0", "F(lambda P+mu Q)=0"),
    ("F1(P;Q)=0", "F(lambda P+mu Q)=0"),
    ("F1(Q;P)=0", "F(lambda P+mu Q)=0"),
    ("F(Q)=0", "F(lambda P+mu Q)=0"),
])
pos = {
    "choose P on S": (-2.7, 1.2),
    "choose Q in plane": (-2.7, -1.2),
    "polar at P": (-0.8, 0.55),
    "polar at Q": (-0.8, -0.55),
    "F(P)=0": (1.0, 1.35),
    "F1(P;Q)=0": (1.0, 0.45),
    "F1(Q;P)=0": (1.0, -0.45),
    "F(Q)=0": (1.0, -1.35),
    "F(lambda P+mu Q)=0": (3.25, 0.0),
}
fig, ax = plt.subplots(figsize=(12, 5.2), constrained_layout=True)
node_colors = ["#E8F1F2" if "choose" in node else "#D7E8BA" if "lambda" in node else "#F9E8D2" for node in G.nodes]
nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=2600, edgecolors="#263238", linewidths=1.2, ax=ax)
nx.draw_networkx_labels(G, pos, font_size=9, ax=ax)
nx.draw_networkx_edges(G, pos, arrows=True, arrowstyle="-|>", arrowsize=16, width=1.4, edge_color="#455A64", ax=ax)
nx.draw_networkx_edge_labels(G, pos, edge_labels={
    ("F(P)=0", "F(lambda P+mu Q)=0"): "lambda^3",
    ("F1(P;Q)=0", "F(lambda P+mu Q)=0"): "lambda^2 mu",
    ("F1(Q;P)=0", "F(lambda P+mu Q)=0"): "lambda mu^2",
    ("F(Q)=0", "F(lambda P+mu Q)=0"): "mu^3",
}, font_size=9, ax=ax)
ax.set_title("Polar coefficients are the four line-containment equations", fontsize=14, weight="bold")
ax.axis("off")
polar_path = FIGURE_ROOT / "polar-line-condition-flow.png"
fig.savefig(polar_path, dpi=170, bbox_inches="tight")
plt.close(fig)
display_artifact(polar_path, width=780)


## 3. Resultant-Degree Intuition

The proof that a line exists can be organized as an elimination problem. After choosing a variable point on a cuspidal tangent section, the condition for a line through that point to lie on the surface becomes a linear equation \(A=0\), a binary quadratic \(B=0\), and a binary cubic \(C=0\). Solving \(A=0\) removes one coordinate. The remaining question is whether the quadratic and cubic share a zero.

For two binary forms of degrees 2 and 3, the Sylvester resultant is a \(5\times 5\) determinant. The source computation tracks the highest powers of the parameter \(\alpha\). The key computational fact is not just that the determinant has high degree, but that the leading degree is 27 and its leading coefficient is nonzero.


In [ ]:
minus_inf = -10**9
resultant_degrees = np.array([
    [1, 5, 9, minus_inf, minus_inf],
    [minus_inf, 1, 5, 9, minus_inf],
    [minus_inf, minus_inf, 1, 5, 9],
    [0, 4, 8, 12, minus_inf],
    [minus_inf, 0, 4, 8, 12],
], dtype=int)
best_degree = -1
best_permutations = []
for perm in itertools.permutations(range(5)):
    if all(resultant_degrees[row, perm[row]] > minus_inf for row in range(5)):
        degree_sum = int(sum(resultant_degrees[row, perm[row]] for row in range(5)))
        if degree_sum > best_degree:
            best_degree = degree_sum
            best_permutations = [perm]
        elif degree_sum == best_degree:
            best_permutations.append(perm)

leading_coeff_matrix = sp.Matrix([
    [-3, 6, -2, 0, 0],
    [0, -3, 6, -2, 0],
    [0, 0, -3, 6, -2],
    [-1, 9, -6, 1, 0],
    [0, -1, 9, -6, 1],
])
leading_coeff_det = sp.Integer(leading_coeff_matrix.det())
assert best_degree == 27
assert leading_coeff_det == 1

fig, ax = plt.subplots(figsize=(8.5, 6), constrained_layout=True)
masked = np.ma.masked_where(resultant_degrees == minus_inf, resultant_degrees)
im = ax.imshow(masked, cmap="viridis", vmin=0, vmax=12)
for i in range(5):
    for j in range(5):
        ax.text(j, i, "0" if resultant_degrees[i, j] == minus_inf else str(resultant_degrees[i, j]),
                ha="center", va="center", color="#666666" if resultant_degrees[i, j] == minus_inf else "white",
                fontsize=10, weight="bold" if resultant_degrees[i, j] != minus_inf else "normal")
ax.set_xticks(range(5), ["col 1", "col 2", "col 3", "col 4", "col 5"])
ax.set_yticks(range(5), ["B row 1", "B row 2", "B row 3", "C row 1", "C row 2"])
ax.set_title("Sylvester resultant degree bookkeeping: maximum alpha-degree is 27", fontsize=13, weight="bold")
ax.text(0.02, -0.14, f"leading coefficient determinant = {leading_coeff_det}; max-degree permutations = {len(best_permutations)}", transform=ax.transAxes, fontsize=10)
fig.colorbar(im, ax=ax, shrink=0.82, label="degree in alpha")
resultant_path = FIGURE_ROOT / "resultant-degree-sylvester.png"
fig.savefig(resultant_path, dpi=170, bbox_inches="tight")
plt.close(fig)
display_artifact(resultant_path, width=720)


## 4. Fermat Cubic Lines as Exact Evidence

The Fermat cubic gives a compact model for all 27 lines. Split the four coordinates into two pairs. For a pair such as \((X,Y)\) and \((Z,T)\), choose roots \(\rho^3=-1\) and \(\sigma^3=-1\), then impose

\[
X=\rho Y,\qquad Z=\sigma T.
\]

Substituting into \(X^3+Y^3+Z^3+T^3\) gives \((\rho^3+1)Y^3+(\sigma^3+1)T^3=0\). There are three pairings of the four coordinates and nine root choices for each pairing, so the list has \(3\cdot 9=27\) candidates. The code verifies that they are distinct lines, not just 27 written equations.


In [ ]:
omega = -sp.Rational(1, 2) + sp.sqrt(3) * sp.I / 2
roots_minus_one = [sp.simplify(-omega**k) for k in range(3)]
root_labels = ["-1", "-omega", "-omega^2"]
coordinate_names = ["X", "Y", "Z", "T"]
pairings = [
    (((0, 1), (2, 3)), "XY|ZT"),
    (((0, 2), (1, 3)), "XZ|YT"),
    (((0, 3), (1, 2)), "XT|YZ"),
]
plucker_pairs = list(itertools.combinations(range(4), 2))


def plucker_coordinates(u, v):
    return tuple(sp.simplify(u[i] * v[j] - u[j] * v[i]) for i, j in plucker_pairs)


def normalize_plucker(p):
    for value in p:
        if sp.simplify(value) != 0:
            return tuple(sp.simplify(item / value) for item in p)
    raise ValueError("zero Plucker vector")


def grassmann_relation(p):
    p01, p02, p03, p12, p13, p23 = p
    return sp.simplify(p01 * p23 - p02 * p13 + p03 * p12)


def incidence_form(p, q):
    p01, p02, p03, p12, p13, p23 = p
    q01, q02, q03, q12, q13, q23 = q
    return sp.simplify(p01 * q23 - p02 * q13 + p03 * q12 + p12 * q03 - p13 * q02 + p23 * q01)


s, t = sp.symbols("s t")
fermat_lines = []
for family_index, (pairs, family_name) in enumerate(pairings):
    (i, j), (k, ell) = pairs
    for a, rho in enumerate(roots_minus_one):
        for b, sigma in enumerate(roots_minus_one):
            u = [sp.Integer(0)] * 4
            v = [sp.Integer(0)] * 4
            u[i] = rho
            u[j] = 1
            v[k] = sigma
            v[ell] = 1
            plucker = plucker_coordinates(u, v)
            normalized = normalize_plucker(plucker)
            coords_on_line = [s * u[idx] + t * v[idx] for idx in range(4)]
            substitution_residual = sp.simplify(sum(coord**3 for coord in coords_on_line))
            line_id = len(fermat_lines) + 1
            equations = f"{coordinate_names[i]}={root_labels[a]} {coordinate_names[j]}, {coordinate_names[k]}={root_labels[b]} {coordinate_names[ell]}"
            fermat_lines.append({
                "id": line_id,
                "family": family_name,
                "family_index": family_index,
                "root_pair": (a, b),
                "equations": equations,
                "u": tuple(u),
                "v": tuple(v),
                "plucker": plucker,
                "normalized_plucker": normalized,
                "substitution_residual": substitution_residual,
            })

unique_pluckers = {tuple(sp.sstr(value) for value in line["normalized_plucker"]) for line in fermat_lines}
assert len(fermat_lines) == 27
assert len(unique_pluckers) == 27
assert all(line["substitution_residual"] == 0 for line in fermat_lines)
assert all(grassmann_relation(line["plucker"]) == 0 for line in fermat_lines)

line_rows = []
for line in fermat_lines:
    line_rows.append({
        "id": line["id"],
        "family": line["family"],
        "root_pair": f"{root_labels[line['root_pair'][0]]}, {root_labels[line['root_pair'][1]]}",
        "equations": line["equations"],
        "line_substitution_residual": str(line["substitution_residual"]),
        "plucker_relation": str(grassmann_relation(line["plucker"])),
    })
line_table_path = TABLE_ROOT / "fermat-lines.csv"
pd.DataFrame(line_rows).to_csv(line_table_path, index=False)

fig, ax = plt.subplots(figsize=(12.5, 4.6), constrained_layout=True)
family_colors = {"XY|ZT": "#0072B2", "XZ|YT": "#D55E00", "XT|YZ": "#009E73"}
for row, (pairs, family_name) in enumerate(pairings):
    ax.text(-0.65, row, family_name, ha="right", va="center", fontsize=12, weight="bold", color=family_colors[family_name])
    for col, (a, b) in enumerate(itertools.product(range(3), repeat=2)):
        rect = plt.Rectangle((col - 0.42, row - 0.34), 0.84, 0.68, facecolor=family_colors[family_name], alpha=0.15, edgecolor=family_colors[family_name], lw=1.5)
        ax.add_patch(rect)
        ax.plot([col - 0.28, col + 0.28], [row - 0.12, row + 0.12], color=family_colors[family_name], lw=2)
        ax.text(col, row + 0.22, f"{root_labels[a]}\n{root_labels[b]}", ha="center", va="center", fontsize=8)
        ax.text(col, row - 0.24, "F|L=0", ha="center", va="center", fontsize=7, color="#263238")
ax.set_xlim(-1.2, 8.7)
ax.set_ylim(2.7, -0.7)
ax.set_xticks(range(9), [f"{a},{b}" for a, b in itertools.product(range(3), repeat=2)], fontsize=8)
ax.set_yticks([])
ax.set_title("Fermat cubic line ledger: 3 coordinate pairings x 9 root choices = 27 lines", fontsize=13, weight="bold")
ax.set_xlabel("root-choice indices for rho^3 = sigma^3 = -1")
for spine in ax.spines.values():
    spine.set_visible(False)
fermat_path = FIGURE_ROOT / "fermat-cubic-lines.png"
fig.savefig(fermat_path, dpi=170, bbox_inches="tight")
plt.close(fig)
display_artifact(fermat_path, width=820)
display_artifact(line_table_path)


## 5. Plucker Incidence Constraints

A projective line spanned by \(u,v\in k^4\) has Plucker coordinates \(p_{ij}=u_i v_j-u_j v_i\). These six coordinates are not arbitrary: they satisfy one quadratic relation defining the Grassmannian \(G(1,3)\subset \mathbf P^5\).

Two lines meet exactly when their two Plucker 2-vectors wedge to zero in \(\bigwedge^4 k^4\). In coordinates, the incidence form is

\[
p_{01}q_{23}-p_{02}q_{13}+p_{03}q_{12}+p_{12}q_{03}-p_{13}q_{02}+p_{23}q_{01}=0.
\]

For the Fermat list this gives a strict combinatorial signature: every line meets 10 of the other 26 lines, so the incidence graph has 27 vertices and 135 edges. The graph is dense enough that labels would obscure the structure, so the static artifact emphasizes families and degree counts, while the HTML lab keeps equation labels in hover text.


In [ ]:
incidence_matrix = np.zeros((27, 27), dtype=int)
for i, line_a in enumerate(fermat_lines):
    for j, line_b in enumerate(fermat_lines):
        if i != j and incidence_form(line_a["plucker"], line_b["plucker"]) == 0:
            incidence_matrix[i, j] = 1
incidence_degrees = incidence_matrix.sum(axis=1).astype(int)
incidence_edges = int(incidence_degrees.sum() // 2)
skew_pairs = int((27 * 26 // 2) - incidence_edges)
assert sorted(set(incidence_degrees.tolist())) == [10]
assert incidence_edges == 135
assert skew_pairs == 216

for line, degree in zip(fermat_lines, incidence_degrees):
    line["incidence_degree"] = int(degree)

G_inc = nx.Graph()
for line in fermat_lines:
    G_inc.add_node(line["id"], family=line["family"], equations=line["equations"])
for i in range(27):
    for j in range(i + 1, 27):
        if incidence_matrix[i, j]:
            G_inc.add_edge(i + 1, j + 1)

pos = nx.spring_layout(G_inc, seed=27, k=0.55, iterations=200)
node_colors = [family_colors[G_inc.nodes[node]["family"]] for node in G_inc.nodes]
fig, axes = plt.subplots(1, 2, figsize=(13, 5.4), gridspec_kw={"width_ratios": [2.5, 1]}, constrained_layout=True)
ax = axes[0]
nx.draw_networkx_edges(G_inc, pos, ax=ax, edge_color="#B0BEC5", width=0.6, alpha=0.38)
nx.draw_networkx_nodes(G_inc, pos, ax=ax, node_color=node_colors, node_size=145, edgecolors="#263238", linewidths=0.6)
for family_name, color in family_colors.items():
    ax.scatter([], [], c=color, label=family_name, s=80)
ax.legend(loc="lower left", frameon=False, fontsize=9)
ax.set_title("Plucker incidence graph for the 27 Fermat lines", fontsize=13, weight="bold")
ax.axis("off")

ax = axes[1]
ax.bar(range(1, 28), incidence_degrees, color="#455A64")
ax.axhline(10, color="#D55E00", lw=2, label="degree 10")
ax.set_ylim(0, 12)
ax.set_xlabel("line id")
ax.set_ylabel("meeting lines")
ax.set_title("uniform incidence degree")
ax.legend(frameon=False)
incidence_path = FIGURE_ROOT / "twenty-seven-line-incidence-graph.png"
fig.savefig(incidence_path, dpi=170, bbox_inches="tight")
plt.close(fig)
display_artifact(incidence_path, width=840)
{"vertices": G_inc.number_of_nodes(), "edges": G_inc.number_of_edges(), "degree_set": sorted(set(incidence_degrees.tolist())), "skew_pairs": skew_pairs}


In [ ]:
def complex_point_to_real_chart(point):
    denom = complex(sp.N(point[3], 15))
    if abs(denom) < 1e-10:
        return None
    values = [complex(sp.N(point[idx] / point[3], 15)) for idx in range(3)]
    return [value.real for value in values]


traces = []
sample_parameters = np.array([-1.6, -1.1, -0.6, -0.25, 0.25, 0.6, 1.1, 1.6])
for line in fermat_lines:
    xs, ys, zs = [], [], []
    for r in sample_parameters:
        point = [sp.N(r * line["u"][idx] + line["v"][idx], 15) for idx in range(4)]
        chart = complex_point_to_real_chart(point)
        if chart is not None:
            xs.append(chart[0]); ys.append(chart[1]); zs.append(chart[2])
    if len(xs) >= 2:
        traces.append(go.Scatter3d(
            x=xs, y=ys, z=zs, mode="lines",
            line=dict(width=5, color=family_colors[line["family"]]), opacity=0.72,
            name=f"L{line['id']:02d} {line['family']}",
            hovertemplate=(
                f"L{line['id']:02d}<br>{line['equations']}<br>"
                f"incidence degree: {line['incidence_degree']}<br>"
                "real chart: Re(X/T), Re(Y/T), Re(Z/T)<extra></extra>"
            ),
            showlegend=False,
        ))

fig = go.Figure(data=traces)
for family_name, color in family_colors.items():
    fig.add_trace(go.Scatter3d(x=[None], y=[None], z=[None], mode="lines", line=dict(color=color, width=6), name=family_name))
fig.update_layout(
    title="Fermat cubic lines: real chart projection of complex representatives",
    scene=dict(xaxis_title="Re(X/T)", yaxis_title="Re(Y/T)", zaxis_title="Re(Z/T)", aspectmode="cube"),
    margin=dict(l=0, r=0, t=48, b=0),
    height=620,
)
html_path = HTML_ROOT / "twenty-seven-line-incidence-graph-lab.html"
fig.write_html(str(html_path), include_plotlyjs=True, full_html=True)
html_text = html_path.read_text(encoding="utf-8")
html_path.write_text("\n".join(line.rstrip() for line in html_text.splitlines()) + "\n", encoding="utf-8")
display_artifact(html_path, height=520)


## 6. A Small Failure Lab

The Fermat construction is fragile in exactly the way the theorem says it should be. If \(\rho^3=-1\), then the pair equation \(X=\rho Y\) cancels the two cubic terms. If we perturb \(\rho\) away from a root of \(-1\), the same two linear equations still define a projective line, but the line no longer lies on the cubic. The residual binary cubic records the failure.

This is a useful habit for algebraic geometry notebooks: every positive construction should have a nearby failure mode. It separates the geometric hypothesis from the drawing convention.


In [ ]:
rho_good = roots_minus_one[0]
sigma_good = roots_minus_one[1]
rho_bad = sp.Rational(2, 3)
good_line_coords = [rho_good * s, s, sigma_good * t, t]
bad_line_coords = [rho_bad * s, s, sigma_good * t, t]
F_fermat = X**3 + Y**3 + Z**3 + T**3
good_residual = sp.simplify(sum(coord**3 for coord in good_line_coords))
bad_residual = sp.factor(sum(coord**3 for coord in bad_line_coords))
assert good_residual == 0
assert bad_residual != 0
failure_lab = {
    "good_root_residual": str(good_residual),
    "bad_root_residual": sp.sstr(bad_residual),
    "interpretation": "The same linear-looking equations stop defining a line on the Fermat cubic when rho^3 != -1.",
}
failure_lab


## 7. Final Sanity Checks

The final checks are specific to this chapter. They verify symbolic identities, geometric incidence data, artifact integrity, and path hygiene. In particular, the JSON files written here use book-local paths (`artifacts/chapter-07/...`) and assert that the stale display-name folder string is absent.


In [ ]:
fermat_gradient = [sp.diff(F_fermat, var) for var in variables]
gradient_common_zero_only_origin = all(grad == 3 * var**2 for grad, var in zip(fermat_gradient, variables))
assert gradient_common_zero_only_origin

chapter_checks = {
    "source_span": {"printed": "109-120", "pdf": "107-118"},
    "line_substitution": {
        "fermat_line_count": len(fermat_lines),
        "unique_plucker_count": len(unique_pluckers),
        "all_fermat_line_residuals_zero": all(line["substitution_residual"] == 0 for line in fermat_lines),
        "bad_root_residual": failure_lab["bad_root_residual"],
    },
    "polar_form": {
        "polar_identity_residual": str(polar_identity_residual),
        "A": sp.sstr(A),
        "B": sp.sstr(B),
        "C": sp.sstr(C),
    },
    "plucker_incidence": {
        "grassmann_relation_all_zero": all(grassmann_relation(line["plucker"]) == 0 for line in fermat_lines),
        "incidence_degree_set": sorted(set(incidence_degrees.tolist())),
        "incidence_edges": incidence_edges,
        "skew_pairs": skew_pairs,
    },
    "resultant_degree": {
        "sylvester_max_degree": best_degree,
        "max_degree_permutation_count": len(best_permutations),
        "leading_coefficient_determinant": str(leading_coeff_det),
    },
    "nonsingularity_evidence": {
        "fermat_gradient_common_projective_zero": False,
        "reason": "In characteristic zero, partials 3X^2, 3Y^2, 3Z^2, 3T^2 vanish simultaneously only at the zero vector.",
    },
}
checks_path = CHECK_ROOT / "line-and-incidence-checks.json"
checks_path.write_text(json.dumps(chapter_checks, indent=2) + "\n", encoding="utf-8")

concept_rows = [
    {"concept": "nonsingular cubic surface", "artifact": "cubic-surface-plane-sections.png", "check": "no doubled residual line in the plane-section model"},
    {"concept": "line substitution on a cubic", "artifact": "fermat-cubic-lines.png", "check": "F(su+tv) has four zero coefficients"},
    {"concept": "polar form equations", "artifact": "polar-line-condition-flow.png", "check": "polar expansion residual is zero"},
    {"concept": "resultant-degree intuition", "artifact": "resultant-degree-sylvester.png", "check": "Sylvester max degree is 27 and leading coefficient is 1"},
    {"concept": "Plucker incidence constraints", "artifact": "twenty-seven-line-incidence-graph.png", "check": "every Fermat line has incidence degree 10"},
    {"concept": "Fermat cubic line count evidence", "artifact": "fermat-lines.csv", "check": "27 distinct normalized Plucker vectors"},
]
concept_table_path = TABLE_ROOT / "concepts.csv"
pd.DataFrame(concept_rows).to_csv(concept_table_path, index=False)

paths = {
    "figures": [plane_section_path, polar_path, resultant_path, fermat_path, incidence_path],
    "html": [html_path],
    "checks": [CHECK_ROOT / "storyboard.json", checks_path],
    "tables": [concept_table_path, line_table_path],
}
validation_summary = validate_chapter_outputs(paths, min_pngs=5)
artifact_summary = {
    "image_stats": validation_summary["image_stats"],
    "artifacts": {key: [book_local(Path(item)) for item in value] for key, value in paths.items()},
    "metrics": {
        "fermat_line_count": len(fermat_lines),
        "unique_plucker_count": len(unique_pluckers),
        "incidence_degree_set": sorted(set(incidence_degrees.tolist())),
        "incidence_edges": incidence_edges,
        "resultant_degree": best_degree,
        "polar_identity_residual": str(polar_identity_residual),
    },
}
artifact_summary_path = CHECK_ROOT / "artifact-summary.json"
artifact_summary_path.write_text(json.dumps(artifact_summary, indent=2) + "\n", encoding="utf-8")
final_sanity = {
    "checks": chapter_checks,
    "artifact_summary": artifact_summary,
    "validation_summary": validation_summary,
}
final_sanity_path = CHECK_ROOT / "final-sanity.json"
final_sanity_path.write_text(json.dumps(final_sanity, indent=2) + "\n", encoding="utf-8")

assert_artifacts([*paths["figures"], *paths["html"], *paths["checks"], *paths["tables"], artifact_summary_path, final_sanity_path])
for stat in validation_summary["image_stats"]:
    assert stat["pixel_std"] > 2.0
assert chapter_checks["line_substitution"]["fermat_line_count"] == 27
assert chapter_checks["line_substitution"]["unique_plucker_count"] == 27
assert chapter_checks["plucker_incidence"]["incidence_degree_set"] == [10]
assert chapter_checks["resultant_degree"]["sylvester_max_degree"] == 27
assert chapter_checks["resultant_degree"]["leading_coefficient_determinant"] == "1"
assert chapter_checks["polar_form"]["polar_identity_residual"] == "0"

stale_token = "Undergraduate Algebraic Geometry"
for json_path in CHECK_ROOT.glob("*.json"):
    assert stale_token not in json_path.read_text(encoding="utf-8"), f"stale display-name path in {json_path.name}"

book_local(final_sanity_path), artifact_summary["metrics"]


## Takeaways

- A line on a cubic surface is an executable object: two homogeneous vectors plus a binary cubic substitution test.
- The polar form is the coefficient bookkeeping that turns line containment into four equations.
- The resultant step explains why a degree-27 polynomial naturally appears when the general proof searches for a line through a moving point on a tangent section.
- The Fermat cubic gives exact evidence for the 27-line count: 3 coordinate pairings, 9 root choices per pairing, and 27 distinct Plucker points.
- Plucker incidence turns the classical configuration into a graph where every line meets 10 others and is skew to 16 others.
- The chapter checks now validate the geometry directly and keep all generated paths book-local under `artifacts/chapter-07/`.
